# Lesson 01: Convolution building blocks

Every CNN (ResNet, UNet, UNet++, SINet ...) is built from small LEGO bricks.
Before we build an encoder or a decoder we need to understand those bricks.

| Part | Topic |
|---|---|
| A | A convolution is just an OpenCV filter (`cv2.filter2D` == `nn.Conv2d`) |
| B | How a convolution changes the tensor **shape** `[B, C, H, W]` |
| C | The standard brick: Conv → BatchNorm → ReLU (`ConvBNReLU`) |
| D | UNet's brick: two of them in a row (`DoubleConv`) |
| E | Other bricks in papers: Residual (ResNet), Dilated (SINet's RF module), Depthwise-separable (MobileNet) |
| F | Look inside: feature maps of a real COD image |

Each part ends with a **✏️ TRY IT** box. Change the code, re-run the cell, watch what changes.

## Setup

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from config import IMAGE_SIZE, find_first_image, get_device

device = get_device()
print(f"device: {device}   image size: {IMAGE_SIZE}")


def count_params(module):
    """Number of trainable numbers (weights) inside a layer / model."""
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

In [ ]:
def show(images, titles=None, cmap=None, cols=4, size=4):
    """Show one or more images inline. Accepts OpenCV BGR (3-channel) or grayscale arrays."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    titles = titles or [""] * len(images)
    rows = (len(images) + cols - 1) // cols
    cols = min(cols, len(images))
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, img, t in zip(axes.flat, images, titles):
        if img.ndim == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # OpenCV is BGR, matplotlib expects RGB
        ax.imshow(img, cmap=cmap or ("gray" if img.ndim == 2 else None))
        ax.set_title(t)
    plt.tight_layout()
    plt.show()

In [ ]:
# Load one image from your dataset (or a synthetic one if the path is wrong).
path = find_first_image()
image = cv2.imread(path) if path else None
if image is not None:
    print(f"Using dataset image: {path}")
else:
    print("Dataset image not found -> using a synthetic image (fix DATASET_ROOT in config.py)")
    image = np.full((IMAGE_SIZE, IMAGE_SIZE, 3), (60, 120, 80), np.uint8)
    image = cv2.add(image, np.random.default_rng(0).integers(0, 40, image.shape, dtype=np.uint8))
    cv2.circle(image, (176, 176), 70, (70, 135, 90), -1)  # a "camouflaged" circle

image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
show([image, gray], ["BGR image", "grayscale"], cols=2)

## Part A: a convolution is an OpenCV filter

In OpenCV you already used filters: blur, Sobel, Canny... A filter is a small matrix (the **kernel**) slid over the image.

`nn.Conv2d` does **exactly** the same thing. The only difference:
- In OpenCV **you** choose the kernel numbers (e.g. Sobel).
- In a CNN the kernel numbers are **learned** during training.

Below we put the Sobel-x numbers into an `nn.Conv2d` and compare it with `cv2.filter2D`.

In [ ]:
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float32)

# --- OpenCV way ---
img = gray.astype(np.float32) / 255.0
edges_cv = cv2.filter2D(img, ddepth=-1, kernel=sobel_x, borderType=cv2.BORDER_CONSTANT)

# --- PyTorch way ---
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    conv.weight[:] = torch.from_numpy(sobel_x)  # put the Sobel numbers into the layer

x = torch.from_numpy(img)[None, None]  # [H, W] -> [1, 1, H, W]  (batch, channel, H, W)
with torch.no_grad():
    edges_torch = conv(x)[0, 0].numpy()

# PyTorch's "convolution" and cv2.filter2D are both cross-correlation, so no kernel flip is needed.
print(f"Input shape (PyTorch): {tuple(x.shape)}   <- [Batch, Channels, Height, Width]")
print(f"conv.weight shape    : {tuple(conv.weight.shape)}   <- [out_ch, in_ch, k, k]")
print(f"Max difference OpenCV vs PyTorch: {np.abs(edges_cv - edges_torch).max():.6f}   (0 = identical)")

show([gray, np.abs(edges_cv), np.abs(edges_torch)], ["input", "cv2.filter2D", "nn.Conv2d"], cols=3)

> ✏️ **TRY IT**
> - Replace `sobel_x` with `sobel_x.T` (Sobel-y) or a blur kernel `np.ones((3, 3), np.float32) / 9`.
> - A CNN layer with `out_channels=64` is simply **64 of these kernels** running in parallel.

## Part B: how Conv2d / pooling change the shape

Output size formula for `Conv2d` (and pooling):

$$H_{out} = \left\lfloor \frac{H_{in} + 2p - d(k-1) - 1}{s} \right\rfloor + 1$$

The 3 cases you will use 95% of the time:

| kernel | stride | padding | result | used for |
|---|---|---|---|---|
| 3 | 1 | 1 | same size | feature extraction |
| 3 | 2 | 1 | half size | downsampling in encoders |
| 1 | 1 | 0 | same size | only changes the number of channels |

In [ ]:
x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)  # one RGB image, 352x352
print(f"input                          : {tuple(x.shape)}")

layers = {
    "Conv k3 s1 p1  (3->64)        ": nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
    "Conv k3 s2 p1  (3->64)        ": nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1),
    "Conv k1 s1 p0  (3->64)        ": nn.Conv2d(3, 64, kernel_size=1),
    "Conv k3 s1 p0  (no padding!)  ": nn.Conv2d(3, 64, kernel_size=3, padding=0),
    "Conv k3 dil2 p2 (dilated)     ": nn.Conv2d(3, 64, kernel_size=3, padding=2, dilation=2),
    "MaxPool k2 s2                 ": nn.MaxPool2d(kernel_size=2, stride=2),
}
for name, layer in layers.items():
    print(f"{name} : {tuple(layer(x).shape)}")

**Key idea:** channels grow (3 → 64 → 128 ...) while H, W shrink. That is exactly what an **encoder** does. A **decoder** does the reverse.

> ✏️ **TRY IT**
> - Add `nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)`. This is the very first layer of ResNet50. What is the output size?
> - Check the formula by hand for the "no padding" row.

## Part C: ConvBNReLU, the standard brick

Conv → BatchNorm → ReLU. The most common brick in all of computer vision.

| part | why |
|---|---|
| Conv | finds patterns (edges, textures ...) |
| BatchNorm | keeps values in a stable range, so training is faster and more stable |
| ReLU | non-linearity; without it, stacking convs is the same as one big conv |

Things **you** can modify (and papers often do): `kernel_size`, `stride`, `dilation`, `norm`, `act`.

In [ ]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, dilation=1,
                 norm="bn", act="relu"):
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2  # keeps H,W the same when stride=1

        # bias=False because BatchNorm already adds its own bias
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=padding,
                              dilation=dilation, bias=(norm is None))

        if norm == "bn":
            self.norm = nn.BatchNorm2d(out_ch)
        elif norm == "gn":  # GroupNorm: better when your batch size is small (1-4)
            self.norm = nn.GroupNorm(num_groups=min(32, out_ch), num_channels=out_ch)
        else:
            self.norm = nn.Identity()

        activations = {
            "relu": nn.ReLU(inplace=True),
            "leakyrelu": nn.LeakyReLU(0.1, inplace=True),
            "gelu": nn.GELU(),
            "silu": nn.SiLU(inplace=True),
            None: nn.Identity(),
        }
        self.act = activations[act]

    def forward(self, x):
        return self.act(self.norm(self.conv(x)))

In [ ]:
block = ConvBNReLU(3, 64)
print(block)

x = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
y = block(x)
print(f"\n{tuple(x.shape)} -> {tuple(y.shape)}   params: {count_params(block):,}")
print(f"After ReLU the minimum value is {y.min().item():.2f} (negatives are cut to 0)")

> ✏️ **TRY IT**
> - `ConvBNReLU(3, 64, act="gelu")` or `norm="gn"`. Does the param count change?
> - `ConvBNReLU(3, 64, act=None)`. What is the minimum value now?

## Part D: DoubleConv, the UNet brick

`ConvBNReLU` × 2. It is used at **every level** of the original UNet, in both the encoder and the decoder.

```
in_ch --[3x3]--> mid_ch --[3x3]--> out_ch
```

Two 3×3 convs see a 5×5 area, with fewer weights than one 5×5 conv (the VGG trick).

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, mid_ch=None):
        super().__init__()
        mid_ch = mid_ch or out_ch
        self.block = nn.Sequential(
            ConvBNReLU(in_ch, mid_ch),
            ConvBNReLU(mid_ch, out_ch),
        )

    def forward(self, x):
        return self.block(x)


block = DoubleConv(3, 64)
x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
print(f"DoubleConv(3, 64): {tuple(x.shape)} -> {tuple(block(x).shape)}   params: {count_params(block):,}")

## Part E: other bricks you will see in papers

**ResidualBlock (ResNet):** `output = F(x) + x`. The `+ x` shortcut lets gradients flow straight back,
so very deep networks (ResNet50 = 50 layers) can still be trained. If the shape changes, the shortcut uses a 1×1 conv to match it.

**Depthwise-separable (MobileNet):** split one 3×3 conv into two cheaper steps:
1. *depthwise*: one 3×3 filter **per channel** (`groups=in_ch`), no mixing
2. *pointwise*: a 1×1 conv that mixes the channels

**Dilated conv:** a 3×3 kernel with gaps, so it sees a bigger area with the same number of weights.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.body = nn.Sequential(
            ConvBNReLU(in_ch, out_ch, stride=stride),
            ConvBNReLU(out_ch, out_ch, act=None),  # no ReLU before the addition
        )
        if stride != 1 or in_ch != out_ch:
            self.shortcut = ConvBNReLU(in_ch, out_ch, kernel_size=1, stride=stride, act=None)
        else:
            self.shortcut = nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.body(x) + self.shortcut(x))


class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, kernel_size=3, padding=1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
        )
        self.pointwise = ConvBNReLU(in_ch, out_ch, kernel_size=1)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

In [ ]:
x = torch.randn(1, 64, 88, 88)
blocks = {
    "ConvBNReLU 3x3 (64->128)  ": ConvBNReLU(64, 128),
    "DoubleConv       (64->128)": DoubleConv(64, 128),
    "ResidualBlock    (64->128)": ResidualBlock(64, 128),
    "ResidualBlock s2 (64->128)": ResidualBlock(64, 128, stride=2),
    "Dilated 3x3 d=3  (64->128)": ConvBNReLU(64, 128, dilation=3),
    "DepthwiseSep     (64->128)": DepthwiseSeparableConv(64, 128),
}
print(f"input: {tuple(x.shape)}")
for name, b in blocks.items():
    print(f"{name}: out {tuple(b(x).shape)}   params {count_params(b):>8,}")

### Receptive field

How big an area of the input one output pixel "sees". For a stack of stride-1 convs:
`RF = 1 + Σ dilation × (kernel − 1)`

**SINet's RF module** runs several dilated branches in parallel, so it can see small **and** large camouflaged objects. We will build it in a later lesson.

In [ ]:
def receptive_field(kernel_sizes, dilations=None):
    dilations = dilations or [1] * len(kernel_sizes)
    rf = 1
    for k, d in zip(kernel_sizes, dilations):
        rf += d * (k - 1)
    return rf


print(f"3 x (3x3) normal         : {receptive_field([3, 3, 3])} px")
print(f"3 x (3x3) dilation 1,3,5 : {receptive_field([3, 3, 3], [1, 3, 5])} px  <- same weights, much bigger view")

> ✏️ **TRY IT**
> - Which block has the fewest params for 64→128? Why?
> - `receptive_field([3, 3, 3, 3], [1, 2, 4, 8])`. How big is it now?

## Part F: look inside, feature maps of a real image

Pass the image through **one random (untrained)** `ConvBNReLU` and show 16 of its 32 output channels.
Each channel is one "detector". Untrained filters already react to edges and colours; training makes them useful.

Note the layout change: OpenCV is `[H, W, C]`, PyTorch is `[B, C, H, W]`.

In [ ]:
torch.manual_seed(0)

rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
x = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(device)  # [H,W,C] -> [1,C,H,W]

block = ConvBNReLU(3, 32).to(device).eval()
with torch.no_grad():
    feats = block(x)[0].cpu().numpy()  # [32, H, W]
print(f"input {tuple(x.shape)} -> features {tuple(feats.shape)}")

show([feats[c] for c in range(16)], [f"channel {c}" for c in range(16)], cmap="viridis", cols=4, size=3)

> ✏️ **TRY IT**
> - Use `ConvBNReLU(3, 32, kernel_size=7)`. Do the feature maps look smoother?
> - Change `torch.manual_seed(0)` to another number. The detectors change, because they are random until trained.

---
**Next lesson (02):** stack these bricks into an **encoder** and print the shape at each stage.